# referencing: 5 Basic Usage Examples

In [ ]:
# Example 1: Using referencing package (safe approach)
import referencing

print('referencing imported successfully')
attrs = [attr for attr in dir(referencing) if not attr.startswith('_')]
print(f'Available attributes: {attrs[:10]}...')

# Test basic functionality
if hasattr(referencing, 'Registry'):
    print('\nTesting Registry:')
    try:
        registry = referencing.Registry()
        print(f'Registry created: {registry}')
    except Exception as e:
        print(f'Registry creation note: {e}')
        raise

if hasattr(referencing, 'Resource'):
    print('\nResource class available')
    
# Show version if available
if hasattr(referencing, '__version__'):
    print(f'\nVersion: {referencing.__version__}')

print('Referencing exploration complete')


In [ ]:
# Example 2: Store a resource in a registry and look it up
from referencing import Registry, Resource
from referencing.jsonschema import DRAFT7

resource_data = {'foo': 'bar', 'type': 'object'}
resource = Resource(contents=resource_data, specification=DRAFT7)
reg = Registry().with_resource('urn:foo', resource)

# Access resource using get() method
res = reg.get('urn:foo')
print("Resource contents:", res.contents)
assert res.contents == resource_data

In [ ]:
# Example 3: Register a JSON Schema resource under its URI
from referencing import Registry, Resource
from referencing.jsonschema import DRAFT7
import json

# Method 1: Using Resource with specification
schema = {"$id": "urn:myschema", "type": "object", "properties": {"name": {"type": "string"}}}
resource = Resource(contents=schema, specification=DRAFT7)
reg = Registry().with_resource('urn:myschema', resource)
print("Schema contents:", reg.get('urn:myschema').contents)
assert reg.get('urn:myschema').contents == schema

In [ ]:
# Example 4: List the URIs held in a registry
from referencing import Registry, Resource
from referencing.jsonschema import DRAFT7

reg = Registry().with_resource('urn:foo', Resource(contents={'foo': 'bar'}, specification=DRAFT7))
print("Registry URIs:", list(reg))
assert list(reg) == ['urn:foo'], list(reg)

In [ ]:
# Example 5: Basic referencing with multiple schemas
from referencing import Registry, Resource
from referencing.jsonschema import DRAFT7

# Create multiple related schemas
user_schema = {
    "$id": "https://example.com/user.json",
    "type": "object",
    "properties": {
        "username": {"type": "string"},
        "profile": {"type": "object"}
    },
    "required": ["username"]
}

config_schema = {
    "$id": "https://example.com/config.json",
    "type": "object", 
    "properties": {
        "theme": {"type": "string", "enum": ["light", "dark"]},
        "notifications": {"type": "boolean"}
    }
}

# Build registry with multiple schemas
registry = Registry()
registry = registry.with_resource("https://example.com/user.json", 
                                 Resource(contents=user_schema, specification=DRAFT7))
registry = registry.with_resource("https://example.com/config.json",
                                 Resource(contents=config_schema, specification=DRAFT7))

# Access and display schemas
print("Available schemas:", list(registry))
user_resource = registry.get("https://example.com/user.json")
print("User required fields:", user_resource.contents.get("required"))
print("Config properties:", list(registry.get("https://example.com/config.json").contents["properties"].keys()))
assert sorted(registry) == ["https://example.com/config.json", "https://example.com/user.json"], list(registry)
assert user_resource.contents.get("required") == ["username"]